In [2]:
import pandas as pd
import numpy as np

In [62]:
houses = pd.read_csv('../Missing Values Imputation/houses.csv',index_col=0).drop(columns=['Servant Quarters'
                                                                                          ,'Kitchens'
                                                                                          ,'Store Rooms'
                                                                                          ,'Storey Unit'])

flats = pd.read_csv('../Missing Values Imputation/flats.csv',index_col=0).drop(columns=['Servant Quarters'
                                                                                          ,'Kitchens'
                                                                                          ,'Store Rooms'])

In [13]:
houses.isnull().sum()

Main Location     0
Price(Cr)         0
Bath(s)           0
Area(Marla)       0
Bedroom(s)        0
Description       0
IsPrimeLoc        0
SolarInstalled    0
WaterBore         0
CornerHouse       0
luxury_type       0
Property era      0
dtype: int64

In [63]:
flats.isnull().sum()

Building             0
Main Location        0
Price(Cr)            0
Bath(s)              0
Area(Marla)          0
Bedroom(s)           0
Description          0
luxury_type          0
Parking Spaces       0
Floor                0
Elevators            0
Floor in Building    0
Price per Unit       0
Property era         0
Floor Level          0
Building Type        0
Elevator Capacity    0
dtype: int64

In [54]:
# Houses

In [19]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Load your dataset
house_pool = houses.copy()

# --- STEP 1: CALCULATE BASE PROPERTY LEVEL METRICS ---
house_pool["Price_Per_Marla"] = (
    house_pool["Price(Cr)"] / house_pool["Area(Marla)"]
)
house_pool["Description_Cleaned"] = (
    house_pool["Description"].fillna("").str.lower()
)

# Convert boolean columns to integers for mathematical averaging
utility_cols = ["SolarInstalled", "WaterBore", "CornerHouse", "IsPrimeLoc"]
house_pool[utility_cols] = house_pool[utility_cols].fillna(0).astype(int)


# --- STEP 2: COLLAPSE HOUSES INTO DISTINCT SOCIETY PROFILES ---

# Group numeric metrics by Main Location (calculating the mean lifestyle profile)
numeric_group_cols = ["Price_Per_Marla", "Area(Marla)", "Bedroom(s)", "Bath(s)"] + utility_cols
society_numeric_profiles = (
    house_pool.groupby("Main Location")[numeric_group_cols].mean()
)

# Concatenate all descriptions in a society together to create a single text pool per location
society_text_profiles = (
    house_pool.groupby("Main Location")["Description_Cleaned"]
    .apply(lambda x: " ".join(x))
    .to_frame()
)

# Merge numeric and text profiles into a master Society Profile DataFrame
society_profiles = society_numeric_profiles.merge(
    society_text_profiles, left_index=True, right_index=True
)


# --- STEP 3: THE LOCATION-TO-LOCATION ENGINE ---
def recommend_similar_societies(selected_location, top_n=2):
    # Standardize input string
    selected_location = str(selected_location).strip()

    if selected_location not in society_profiles.index:
        return f"Location '{selected_location}' not found in the database. Available options: {list(society_profiles.index)}"

    # Isolate the target society profile row
    target_society = society_profiles.loc[[selected_location]]

    # Exclude the target society itself from the matching pool to discover alternatives
    alternative_societies = society_profiles[
        society_profiles.index != selected_location
    ].copy()

    if alternative_societies.empty:
        return "No alternative societies available in the dataset for comparison."

    # PASS 1: Calculate Numeric & Infrastructure Similarity
    scaler = MinMaxScaler()
    config_cols = [
        "Price_Per_Marla",
        "Area(Marla)",
        "Bedroom(s)",
        "Bath(s)",
        "SolarInstalled",
        "WaterBore",
    ]

    scaled_alt_societies = scaler.fit_transform(alternative_societies[config_cols])
    scaled_target_society = scaler.transform(target_society[config_cols])

    sim_numeric = cosine_similarity(
        scaled_target_society, scaled_alt_societies
    ).flatten()

    # PASS 2: Calculate Lifestyle/Luxury Vibe Similarity via TF-IDF Text Mining
    luxury_vocabulary = [
        "double unit",
        "single unit",
        "brand new",
        "designer",
        "park face",
        "open view",
        "gated community",
        "imported fittings",
        "luxury",
    ]

    tfidf = TfidfVectorizer(vocabulary=luxury_vocabulary)
    tfidf_alt = tfidf.fit_transform(alternative_societies["Description_Cleaned"])
    tfidf_target = tfidf.transform(target_society["Description_Cleaned"])

    sim_text = cosine_similarity(tfidf_target, tfidf_alt).flatten()

    # PASS 3: Combine Scores (Numerical Specs: 60%, Text Luxury Vibe: 40%)
    final_scores = (sim_numeric * 0.60) + (sim_text * 0.40)

    # Attach scores and rank
    alternative_societies["similarity_score"] = final_scores
    recommendations = alternative_societies.sort_values(
        by="similarity_score", ascending=False
    )

    # Format output for clean visualization
    return recommendations[
        [
            "Price_Per_Marla",
            "Area(Marla)",
            "SolarInstalled",
            "WaterBore",
            "similarity_score",
        ]
    ].head(top_n)


# --- RUN TEST QUERY ---
# If a user types "D-12", what other societies offer a similar pricing model and lifestyle tier?
recommend_similar_societies(selected_location="D-12", top_n=2)

,Price_Per_Marla,Area(Marla),SolarInstalled,WaterBore,similarity_score
Main Location,,,,,
G-11,0.883213,9.738182,0.018182,0.218182,0.987631
E-11,0.785548,13.370313,0.046875,0.093750,0.962208


In [44]:
df = recommend_similar_societies(selected_location="Taramrri", top_n=2)
df = df.rename(columns={'Main Location':'Recommended Location','Price_Per_Marla':"Avg Price per Marla",'Area(Marla)':"Avg Area(Marlas)"})
df

,Avg Price per Marla,Avg Area(Marlas),SolarInstalled,WaterBore,similarity_score
Main Location,,,,,
Chatha Bakhtawar,0.336667,6.333333,0.0,1.0,0.596233
Alipur Farash,0.180000,5.000000,0.0,1.0,0.589300


In [52]:
df.rename_axis(index={"Main Location":'dfdfd'},inplace=True)

In [53]:
df

,Avg Price per Marla,Avg Area(Marlas),SolarInstalled,WaterBore,similarity_score
dfdfd,,,,,
Chatha Bakhtawar,0.336667,6.333333,0.0,1.0,0.596233
Alipur Farash,0.180000,5.000000,0.0,1.0,0.589300


In [51]:
df.index

Index(['Chatha Bakhtawar', 'Alipur Farash'], dtype='str', name='Main Location')

In [37]:
houses['Main Location'].str.contains("Main Mar").sum()

np.int64(0)

In [24]:
houses['Main Location'].unique().tolist()

['D-12',
 'Bahria Enclave',
 'FECHS',
 'Faisal Town Phase 1',
 'G-13',
 'MPCHS - Multi Gardens',
 'B-17',
 'D-17',
 'CBR Town Phase 1',
 'Soan Garden',
 'DHA Phase 2',
 'E-7',
 'G-11',
 'Naval Anchorage',
 'G-9',
 'Margalla View Housing Society',
 'Bahria Town',
 'Ghauri Town',
 'Mumtaz City',
 'Bani Gala',
 'G-14',
 'I-14',
 'G-8',
 'F-8',
 'F-17',
 'Margalla Town',
 'E-11',
 'DHA Valley',
 'G-15',
 'F-10',
 'I-11',
 'DHA Phase 1',
 'F-7',
 'I-8',
 'H-13',
 'Zaraj Housing Scheme',
 'Park View City',
 'I-10',
 'Gulberg Greens',
 'DHA Phase 5',
 'Gulberg Residencia',
 'G-10',
 'Kuri Road',
 'F-11',
 'Top City 1',
 'Park Enclave',
 'Khanna Pul',
 'Jhangi Syedan',
 'PWD Housing Scheme',
 'E-18',
 'Bhara kahu',
 'Jinnah Gardens',
 'G-16',
 'F-6',
 'Emaar Canyon Views',
 'Korang Town',
 'Al Qaim Town',
 'National Police Foundation O-9',
 'Airport Green Garden',
 'C-18',
 'Taramrri',
 'Pakistan Town',
 'Faisal Hills',
 'Tarnol',
 'G-12',
 'Green Avenue',
 'Burma Town',
 'Arsalan Town',
 'Pir

In [55]:
# Flats

In [56]:
flats.columns

Index(['Bath(s)', 'Area(Marla)', 'Bedroom(s)', 'Servant Quarters', 'Kitchens',
       'Store Rooms', 'Building', 'Main Location', 'luxury_type',
       'Parking Spaces', 'Floor', 'Elevators', 'Floor in Building',
       'Price per Unit', 'Property era', 'Floor Level', 'Building Type',
       'Elevator Capacity'],
      dtype='str')

In [73]:
# ohe = OneHotEncoder

ohe_col = [
    'Building Type',
    'Elevator Capacity',
]

In [76]:
pd.get_dummies(flats,columns=ohe_col)

,Building,Main Location,Price(Cr),Bath(s),Area(Marla),Bedroom(s),Description,luxury_type,Parking Spaces,Floor,...,Floor Level,Building Type_Ground/Flat,Building Type_High Rise,Building Type_Lower Levels,Building Type_Mid Rise Core,Building Type_Skyscarper,Elevator Capacity_Dual Setup,Elevator Capacity_High-Capacity,Elevator Capacity_Multiple Bank,Elevator Capacity_Single/None
7,Diplomatic Enclave,Diplomatic Enclave,7.20,2.0,6.5,2.0,Luxurious 2-Bedroom Semi-Furnished Apartment f...,0,1.0,1.0,...,Ground/First Floor,False,False,False,True,False,False,False,False,True
8,Karakoram Diplomatic Enclave,Karakoram Diplomatic Enclave,13.25,4.0,11.8,3.0,Luxurious 3-Bedroom Fully Furnished Corner Apa...,1,1.0,1.0,...,Ground/First Floor,False,False,False,True,False,False,False,False,True
9,Diplomatic Enclave,Diplomatic Enclave,5.40,3.0,7.3,3.0,Details - 3 spacious bedrooms with modern amen...,1,1.0,1.0,...,Ground/First Floor,False,False,False,True,False,False,False,False,True
15,Cube Apartments,Bahria Enclave,1.04,2.0,5.0,1.0,Cube Apartment One Bedroom Apartment Beautiful...,0,1.0,2.0,...,Lower Floors,False,False,False,True,False,False,False,True,False
17,The Centaurus,F-8,5.75,1.0,3.9,1.0,Note: Dealers and intermediaries are requested...,0,1.0,1.0,...,Ground/First Floor,True,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6474,Zaraj Housing Scheme,Zaraj Housing Scheme,1.30,2.0,4.3,2.0,Perfect Prime Location 850 Square Feet Flat In...,0,1.0,3.0,...,Lower Floors,False,False,False,True,False,True,False,False,False
6475,El Cielo,GT Road,1.35,3.0,6.3,3.0,El Cielo is another jewel in the Crown of Down...,0,1.0,8.0,...,Mid-Level,False,False,False,True,False,False,False,False,True
6476,Askari Heights 4,DHA Phase 5,3.00,3.0,10.0,3.0,3 Bed Flat For Sale In Askari Heights 4 Reason...,0,1.0,1.0,...,Ground/First Floor,True,False,False,False,False,False,False,False,True
6477,Askari Heights 4,DHA Phase 5,2.85,3.0,10.0,3.0,A prime location like Askari Heights 4 is the ...,0,1.0,1.0,...,Ground/First Floor,True,False,False,False,False,False,False,False,True


In [81]:
flat_pool[''Elevator Capacity_Single/None'']

SyntaxError: invalid syntax. Perhaps you forgot a comma? (129180107.py, line 1)

In [87]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# --- STEP 1: CALCULATE BASE FLAT METRICS & DUMMIES ---
# Safely create one-hot dummy variables right at the beginning
flat_pool = pd.get_dummies(flats, columns=ohe_col)

flat_pool["Price_Per_Marla"] = (
    flat_pool["Price(Cr)"] / flat_pool["Area(Marla)"]
)

flat_pool["Description_Cleaned"] = (
    flat_pool["Description"].fillna("").str.lower()
)

# Core structural, luxury, and encoded columns unique to vertical complexes
flat_utility_cols = [
    "Elevators",
    "Floor in Building",
    "Building Type_Ground/Flat", "Building Type_High Rise",
    "Building Type_Lower Levels", "Building Type_Mid Rise Core",
    "Building Type_Skyscarper", "Elevator Capacity_Dual Setup",
    "Elevator Capacity_High-Capacity", "Elevator Capacity_Multiple Bank",
    "Elevator Capacity_Single/None",
    "Parking Spaces"
]

# Convert features to numeric and handle missing values safely
flat_pool[flat_utility_cols] = flat_pool[flat_utility_cols].fillna(0).astype(float)


# --- STEP 2: COLLAPSE FLATS INTO DISTINCT SOCIETY PROFILES ---

# Group numerical features to compute the average apartment standard per building cluster
flat_numeric_group_cols = ["Price_Per_Marla", "Area(Marla)", "Bedroom(s)", "Bath(s)"] + flat_utility_cols
society_flat_numeric = (
    flat_pool.groupby("Building")[flat_numeric_group_cols].mean()
)

# Combine all apartment descriptions within a building cluster into a single text block
society_flat_text = (
    flat_pool.groupby("Building")["Description_Cleaned"]
    .apply(lambda x: " ".join(x))
    .to_frame()
)

# Merge numeric profiles and text blocks into a master Flat Society Profile
flat_society_profiles = society_flat_numeric.merge(
    society_flat_text, left_index=True, right_index=True
)


# --- STEP 3: THE LOCATION-TO-LOCATION FLAT ENGINE ---
def recommend_similar_flat_societies(selected_location, top_n=2):
    selected_location = str(selected_location).strip()

    if selected_location not in flat_society_profiles.index:
        return f"Location '{selected_location}' has no flat listings. Options: {list(flat_society_profiles.index)}"

    # Isolate the target society profile
    target_profile = flat_society_profiles.loc[[selected_location]]

    # Exclude the target society to ensure we recommend alternative locations
    alternative_profiles = flat_society_profiles[
        flat_society_profiles.index != selected_location
    ].copy()

    if alternative_profiles.empty:
        return "No alternative locations containing flat profiles were found."

    # PASS 1: Calculate Spatial, Structural, & Building Utility Similarity
    scaler = MinMaxScaler()
    
    # FIXED: Ensure your dummy features are actually included in the similarity calculation
    config_cols = [
        "Price_Per_Marla",
        "Area(Marla)",
        "Bedroom(s)",
        "Bath(s)",
    ] + flat_utility_cols

    scaled_alt = scaler.fit_transform(alternative_profiles[config_cols])
    
    # FIXED: Wrapped row cleanly to avoid dimensional scikit-learn warnings/errors
    target_df = pd.DataFrame(target_profile[config_cols].values, columns=config_cols)
    scaled_target = scaler.transform(target_df)

    sim_numeric = cosine_similarity(scaled_target, scaled_alt).flatten()

    # PASS 2: Extract High-Rise Luxury Vibe (TF-IDF Text Mining)
    flat_vocabulary = [
        "penthouse", "studio", "luxury apartment", "standby generator",
        "reception lobby", "cctv security", "dedicated parking",
        "fast elevators", "monthly rent", "brand new",
    ]

    tfidf = TfidfVectorizer(vocabulary=flat_vocabulary)
    tfidf_alt = tfidf.fit_transform(alternative_profiles["Description_Cleaned"])
    tfidf_target = tfidf.transform(target_profile["Description_Cleaned"])

    sim_text = cosine_similarity(tfidf_target, tfidf_alt).flatten()

    # PASS 3: Combined Weighted Score (Specs/Infrastructure: 60%, Text Vibe: 40%)
    final_scores = (sim_numeric * 0.60) + (sim_text * 0.40)

    # Attach scores and rank locations
    alternative_profiles["similarity_score"] = final_scores
    recommendations = alternative_profiles.sort_values(
        by="similarity_score", ascending=False
    )

    return recommendations[
        [
            "Price_Per_Marla",
            "Area(Marla)",
            "Elevators",
            "similarity_score",
        ]
    ].head(top_n)


# --- RUN TEST QUERY ---
recommend_similar_flat_societies(selected_location="The Centaurus", top_n=2)

,Price_Per_Marla,Area(Marla),Elevators,similarity_score
Building,,,,
Silver Oaks Apartments,0.876097,6.925,4.333333,0.858348
Cube Apartments,0.249651,5.300,3.384615,0.816031


In [88]:
recommend_similar_flat_societies(selected_location="The Centaurus", top_n=5)

,Price_Per_Marla,Area(Marla),Elevators,similarity_score
Building,,,,
Silver Oaks Apartments,0.876097,6.925000,4.333333,0.858348
Cube Apartments,0.249651,5.300000,3.384615,0.816031
Blue Area,1.111163,4.375000,3.000000,0.808441
Top City 1,0.313452,3.647541,1.639344,0.760788
DHA Phase 5,0.429360,5.812500,1.125000,0.754318


In [69]:
flats['Building'].unique().tolist()

['Diplomatic Enclave',
 'Karakoram Diplomatic Enclave',
 'Cube Apartments',
 'The Centaurus',
 'Gulberg Greens',
 'Faisal Town Phase 1',
 'F-11',
 'El Cielo',
 'Residency',
 'Eighteen',
 'Smama Star Mall & Residency',
 'E-11',
 'F-17',
 'DHA Phase 2',
 'Capital Heights',
 'Top City 1',
 'PHAF Officers Residencia',
 'DHA Phase 5',
 'Park View City',
 'Ghauri Town',
 'DHA Phase 4',
 'Al-Ghurair Giga',
 'Goldcrest Views',
 'Askari Heights 4',
 'MPCHS',
 'B-17',
 'Faisal Town',
 'Soan Garden',
 'Sukh Chayn Residence',
 'Faisal Hills',
 'Constitution Avenue',
 'Pine Heights Luxury Apartments',
 'G-10',
 'Askari Tower 3',
 'Bahria Enclave',
 'Diamond Mall & Residency',
 'Warda Hamna Residencia 3',
 'G-15',
 'F-8',
 'F-10',
 'The Arch',
 'Executive Heights',
 'Executive Apartments',
 'Gulberg Residencia',
 'G-11',
 'Silver Oaks Apartments',
 'Mall of Islamabad',
 'Skypark One',
 'PAF Tarnol',
 'Zarkon Heights',
 'Gulberg Heights',
 'Bani Gala',
 'Robinas Luxury Residences',
 'The Royal Mall a